# Retrieval smoke test

Top-5 results from each retrieval leg (dense / BM25) for 10 hand-picked
queries — three monolingual per language plus deliberately cross-lingual ones
(Ukrainian question about an English regulation, etc.). Sanity check, not an
evaluation: the eval harness with metrics lands in Phase 8.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # make `src` importable from notebooks/

In [2]:
from src.ingestion.chunker import load_chunks
from src.retrieval.bm25_index import BM25Index
from src.retrieval.embedding import get_embedder
from src.retrieval import qdrant_store

chunks = {c.chunk_id: c for c in load_chunks("child")}
bm25 = BM25Index.load()
embedder = get_embedder()
client = qdrant_store.get_client()
print("chunks:", len(chunks))

/Users/ivantomilo/Developer/personal/multilungual-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


chunks: 3710


In [3]:
QUERIES = [
    ("en", "What are the lawful bases for processing personal data?"),
    ("en", "Which AI practices are prohibited under the AI Act?"),
    ("en", "What obligations do very large online platforms have?"),
    ("pl", "Jakie kary pieniężne przewiduje RODO za naruszenia?"),
    ("pl", "Ile dni urlopu wypoczynkowego przysługuje pracownikowi?"),
    ("pl", "Czy można trenować modele AI na danych osobowych?"),
    ("uk", "Які права має суб'єкт персональних даних?"),
    # deliberately cross-lingual: answers live in another language's documents
    ("uk", "Які системи штучного інтелекту заборонені в ЄС?"),
    ("pl", "Jakie wymogi muszą spełniać systemy AI wysokiego ryzyka?"),
    ("en", "What does Ukrainian law require for consent to data processing?"),
]

def show(results, label):
    print(f"  {label}")
    for cid, score in results[:5]:
        c = chunks[cid]
        print(f"    {score:6.3f}  [{c.language}] {c.source_id:24} {c.ref or c.kind}")

for lang, q in QUERIES:
    print(f"\n=== [{lang}] {q}")
    dense = qdrant_store.search(client, embedder.embed_query(q), top_k=5)
    show(dense, "dense (e5 + qdrant)")
    show(bm25.search(q, top_k=5), "bm25")


=== [en] What are the lawful bases for processing personal data?


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5007.31it/s]

  dense (e5 + qdrant)
     0.878  [en] gdpr_en                  Recital 46
     0.867  [en] gdpr_en                  Article 6
     0.867  [en] gdpr_en                  Recital 40
     0.867  [en] gdpr_en                  Recital 50
     0.865  [en] gdpr_en                  Recital 47
  bm25
    10.880  [en] gdpr_en                  Recital 39
     9.213  [en] gdpr_en                  Recital 39
     9.197  [en] gdpr_en                  Recital 50
     9.052  [en] gdpr_en                  Recital 10
     8.539  [en] gdpr_en                  Article 6

=== [en] Which AI practices are prohibited under the AI Act?
  dense (e5 + qdrant)
     0.876  [en] eu_ai_act_2024           Article 5
     0.859  [en] eu_ai_act_2024           Recital 29
     0.850  [en] eu_ai_act_2024           Article 5
     0.844  [en] eu_ai_act_2024           Recital 28
     0.837  [en] eu_ai_act_2024           Article 5
  bm25
     9.267  [en] eu_ai_act_2024           Recital 54
     8.980  [en] eu_ai_act_2024      

  dense (e5 + qdrant)
     0.892  [en] dsa_en                   Recital 75
     0.863  [en] dsa_en                   Recital 65
     0.860  [en] dsa_en                   Recital 48
     0.860  [en] dsa_en                   Recital 137
     0.854  [en] dsa_en                   Recital 76
  bm25
     9.918  [en] dsa_en                   Recital 75
     9.832  [en] dsa_en                   Recital 65
     9.693  [en] dsa_en                   Article 32
     9.394  [en] dsa_en                   Recital 108
     9.283  [en] dsa_en                   Recital 76

=== [pl] Jakie kary pieniężne przewiduje RODO za naruszenia?
  dense (e5 + qdrant)
     0.882  [pl] edpb_opinion_28_2024_pl  pkt 113
     0.856  [pl] gdpr_pl                  Art. 83
     0.853  [pl] pl_data_protection_act   Art. 1
     0.852  [pl] gdpr_pl                  Motyw 146
     0.852  [pl] gdpr_pl                  Motyw 148
  bm25
     8.477  [pl] gdpr_pl                  Art. 83
     8.282  [pl] pl_data_protection_act   Art

  dense (e5 + qdrant)
     0.883  [pl] pl_labour_code           Art. 167²
     0.882  [pl] pl_labour_code           Art. 14
     0.882  [pl] pl_labour_code           Art. 162
     0.877  [pl] pl_labour_code           Art. 154
     0.872  [pl] pl_labour_code           Art. 152
  bm25
    10.747  [pl] pl_labour_code           Art. 152
     9.028  [pl] pl_labour_code           Art. 29
     7.279  [pl] pl_labour_code           Art. 282
     6.666  [pl] pl_labour_code           Art. 180
     6.663  [pl] pl_labour_code           Art. 173¹

=== [pl] Czy można trenować modele AI na danych osobowych?


  dense (e5 + qdrant)
     0.878  [pl] edpb_opinion_28_2024_pl  pkt 29
     0.875  [pl] edpb_opinion_28_2024_pl  pkt 29
     0.873  [pl] edpb_opinion_28_2024_pl  pkt 31
     0.870  [pl] edpb_opinion_28_2024_pl  pkt 28
     0.867  [pl] edpb_opinion_28_2024_pl  pkt 4
  bm25
     9.421  [pl] edpb_opinion_28_2024_pl  pkt 34
     8.799  [pl] edpb_opinion_28_2024_pl  pkt 38
     8.608  [pl] edpb_opinion_28_2024_pl  pkt 29
     7.183  [pl] edpb_opinion_28_2024_pl  pkt 29
     7.101  [pl] edpb_opinion_28_2024_pl  pkt 4

=== [uk] Які права має суб'єкт персональних даних?
  dense (e5 + qdrant)
     0.903  [uk] ua_data_protection_law   Стаття 8
     0.870  [uk] ua_data_protection_law   Стаття 2
     0.864  [uk] ua_data_protection_law   Стаття 12
     0.864  [uk] ua_data_protection_law   Стаття 16
     0.863  [uk] ua_data_protection_law   Стаття 2
  bm25
    22.394  [uk] ua_data_protection_law   Стаття 8
    12.680  [uk] ua_data_protection_law   Стаття 4
    12.314  [uk] ua_data_protection_law   С

  dense (e5 + qdrant)
     0.830  [en] gdpr_en                  Recital 40
     0.821  [en] dsa_en                   Recital 10
     0.819  [en] gdpr_en                  Recital 45
     0.818  [en] gdpr_en                  Article 6
     0.815  [en] eu_ai_act_2024           Recital 94
  bm25
     8.360  [en] gdpr_en                  Recital 43
     7.839  [en] gdpr_en                  Recital 45
     7.524  [en] gdpr_en                  Article 7
     7.486  [en] gdpr_en                  Recital 10
     7.438  [en] eu_ai_act_2024           Recital 141
